# MLP Surrogate Ablation Study for Fisher-Regularized Variance Dynamics

This Colab notebook computes the MLP ablation table for the paper:

**Fisher-Regularized Wasserstein Dynamics with Neural Surrogate Learning for Transient Distributional Relaxation**

It trains neural surrogate models for the reduced Gaussian-manifold variance dynamics

$$\dot{u}=2(1-u)+\frac{\varepsilon}{u},\qquad u(0)=u_0,$$

and reports four metrics:

- $E_{\mathrm{traj}}$: trajectory mean-squared error
- $E_{\mathrm{cross}}$: crossing-time error at $u=1$
- $E_{\mathrm{over}}$: maximum-variance error
- $E_{\mathrm{eq}}$: final equilibrium error relative to $u^*_{\varepsilon}$

Run all cells in Google Colab. The final cell prints a LaTeX table for Overleaf.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.precision", 6)

## 2. Fisher-regularized variance dynamics

The reduced ODE is

$$\dot{u}=2(1-u)+\frac{\varepsilon}{u}.$$

The displaced equilibrium is

$$u^*_{\varepsilon}=\frac{1+\sqrt{1+2\varepsilon}}{2}.$$

In [2]:
def equilibrium(eps: float) -> float:
    return (1.0 + np.sqrt(1.0 + 2.0 * eps)) / 2.0


def simulate_variance(u0: float, eps: float, t_grid: np.ndarray) -> np.ndarray:
    def rhs(t, y):
        u = max(float(y[0]), 1e-12)
        return [2.0 * (1.0 - u) + eps / u]
    sol = solve_ivp(
        rhs,
        (float(t_grid[0]), float(t_grid[-1])),
        [float(u0)],
        t_eval=t_grid,
        rtol=1e-9,
        atol=1e-11,
    )
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol.y[0]


def crossing_time(values: np.ndarray, t_grid: np.ndarray, level: float = 1.0):
    s = values - level
    for k in range(1, len(values)):
        if (s[k - 1] <= 0 and s[k] >= 0) or (s[k - 1] >= 0 and s[k] <= 0):
            if values[k] == values[k - 1]:
                return float(t_grid[k])
            return float(
                t_grid[k - 1]
                + (level - values[k - 1])
                * (t_grid[k] - t_grid[k - 1])
                / (values[k] - values[k - 1])
            )
    return np.nan

## 3. Generate train/test datasets

The split is over parameter pairs $(u_0,arepsilon)$, not individual time points.

In [3]:
T_MAX = 5.0
N_TIME = 101
T_GRID = np.linspace(0.0, T_MAX, N_TIME)

TRAIN_U0_RANGE = (0.2, 2.5)
TRAIN_EPS_RANGE = (0.01, 0.5)

# Increase these for final-quality runs; decrease if Colab runtime is slow.
N_TRAIN_PAIRS = 1200
N_TEST_PAIRS = 300

TRAIN_SEED = 123
TEST_SEED = 456


def make_dataset(n_pairs, u0_range, eps_range, t_grid, seed):
    rng = np.random.default_rng(seed)
    u0s = rng.uniform(u0_range[0], u0_range[1], n_pairs)
    epss = rng.uniform(eps_range[0], eps_range[1], n_pairs)
    X, y, trajectories = [], [], []
    for u0, eps in zip(u0s, epss):
        u = simulate_variance(u0, eps, t_grid)
        trajectories.append((float(u0), float(eps), u))
        for t, uj in zip(t_grid, u):
            X.append([u0, eps, t])
            y.append(uj)
    return np.asarray(X), np.asarray(y), trajectories


X_train, y_train, train_traj = make_dataset(
    N_TRAIN_PAIRS, TRAIN_U0_RANGE, TRAIN_EPS_RANGE, T_GRID, TRAIN_SEED
)
X_test, y_test, test_traj = make_dataset(
    N_TEST_PAIRS, TRAIN_U0_RANGE, TRAIN_EPS_RANGE, T_GRID, TEST_SEED
)

print("Training samples:", X_train.shape, y_train.shape)
print("Test samples:", X_test.shape, y_test.shape)
print("Training trajectories:", len(train_traj))
print("Test trajectories:", len(test_traj))

Training samples: (121200, 3) (121200,)
Test samples: (30300, 3) (30300,)
Training trajectories: 1200
Test trajectories: 300


## 4. Define mechanism-aware metrics

In [4]:
def evaluate_model(model, trajectories, t_grid, t_max=T_MAX):
    traj_errs, cross_errs, over_errs, eq_errs = [], [], [], []
    for u0, eps, true_u in trajectories:
        X = np.column_stack([
            np.full_like(t_grid, u0),
            np.full_like(t_grid, eps),
            t_grid,
        ])
        pred_u = model.predict(X)
        traj_errs.append(np.mean((pred_u - true_u) ** 2))
        true_cross = crossing_time(true_u, t_grid, level=1.0)
        pred_cross = crossing_time(pred_u, t_grid, level=1.0)
        if np.isnan(true_cross) and np.isnan(pred_cross):
            pass
        elif np.isnan(true_cross) or np.isnan(pred_cross):
            cross_errs.append(t_max)
        else:
            cross_errs.append(abs(pred_cross - true_cross))
        over_errs.append(abs(np.max(pred_u) - np.max(true_u)))
        eq_errs.append(abs(pred_u[-1] - equilibrium(eps)))
    return {
        "E_traj": float(np.mean(traj_errs)),
        "E_cross": float(np.mean(cross_errs)) if cross_errs else np.nan,
        "E_over": float(np.mean(over_errs)),
        "E_eq": float(np.mean(eq_errs)),
    }

In [7]:
import numpy as np

class ResidualOUSurrogate:
    """
    Mechanism-aware residual surrogate:
        u_hat(t; u0, eps) = u_OU(t; u0) + eps * r_theta(u0, eps, t)

    where
        u_OU(t; u0) = 1 + (u0 - 1) exp(-2t).
    """

    def __init__(
        self,
        hidden_layers=(64, 64),
        input_scaling=True,
        target_scaling=True,
        random_state=0,
    ):
        self.hidden_layers = hidden_layers
        self.input_scaling = input_scaling
        self.target_scaling = target_scaling
        self.random_state = random_state

        self.model = build_mlp(
            hidden_layers=hidden_layers,
            input_scaling=input_scaling,
            target_scaling=target_scaling,
            random_state=random_state,
        )

    @staticmethod
    def u_ou(X):
        """
        X columns are assumed to be:
            X[:, 0] = u0
            X[:, 1] = eps
            X[:, 2] = t
        """
        X = np.asarray(X)
        u0 = X[:, 0]
        t = X[:, 2]
        return 1.0 + (u0 - 1.0) * np.exp(-2.0 * t)

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).ravel()

        eps = np.maximum(X[:, 1], 1e-12)
        y_residual = (y - self.u_ou(X)) / eps

        self.model.fit(X, y_residual)
        return self

    def predict(self, X):
        X = np.asarray(X)

        eps = X[:, 1]
        residual_pred = self.model.predict(X)

        return self.u_ou(X) + eps * residual_pred

## 5. Train ablation variants

Variants:

- Small MLP $(32)$
- Medium MLP $(64,64)$
- Large MLP $(128,128)$
- Medium MLP without input scaling
- Medium MLP without target scaling

The medium MLP corresponds to the main configuration used in the paper.

In [8]:
def build_mlp(hidden_layers, input_scaling=True, target_scaling=True, random_state=0):
    steps = []
    if input_scaling:
        steps.append(("input_scaler", StandardScaler()))
    steps.append((
        "mlp",
        MLPRegressor(
            hidden_layer_sizes=hidden_layers,
            activation="relu",
            solver="adam",
            learning_rate_init=5e-4,
            batch_size=512,
            max_iter=1500,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=60,
            tol=1e-8,
            random_state=random_state,
        ),
    ))
    pipe = Pipeline(steps)
    if target_scaling:
        return TransformedTargetRegressor(regressor=pipe, transformer=StandardScaler())
    return pipe


ablation_variants = [
    ("Small MLP $(32)$", "direct", (32,), True, True, 11),
    ("Medium MLP $(64,64)$", "direct", (64, 64), True, True, 12),
    ("Residual MLP $(64,64)$", "residual", (64, 64), True, True, 16),
    ("Large MLP $(128,128)$", "direct", (128, 128), True, True, 13),
    ("Medium MLP w/o input scaling", "direct", (64, 64), False, True, 14),
    ("Medium MLP w/o target scaling", "direct", (64, 64), True, False, 15),
]

rows = []
trained_models = {}

for name, model_type, layers, input_scaling, target_scaling, seed in ablation_variants:
    print(f"Training: {name}")

    if model_type == "direct":
        model = build_mlp(
            layers,
            input_scaling=input_scaling,
            target_scaling=target_scaling,
            random_state=seed,
        )
    elif model_type == "residual":
        model = ResidualOUSurrogate(
            hidden_layers=layers,
            input_scaling=input_scaling,
            target_scaling=target_scaling,
            random_state=seed,
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    model.fit(X_train, y_train)
    scores = evaluate_model(model, test_traj, T_GRID)

    rows.append([
        name,
        scores["E_traj"],
        scores["E_cross"],
        scores["E_over"],
        scores["E_eq"],
    ])

    trained_models[name] = model

ablation_df = pd.DataFrame(
    rows,
    columns=["Variant", "E_traj", "E_cross", "E_over", "E_eq"],
)

ablation_df

Training: Small MLP $(32)$
Training: Medium MLP $(64,64)$
Training: Residual MLP $(64,64)$
Training: Large MLP $(128,128)$
Training: Medium MLP w/o input scaling
Training: Medium MLP w/o target scaling


,Variant,E_traj,E_cross,E_over,E_eq
0,Small MLP $(32)$,5.450561e-06,0.057479,0.002974,0.001132
1,"Medium MLP $(64,64)$",7.150012e-07,0.004636,0.001022,0.000356
2,"Residual MLP $(64,64)$",2.216261e-08,0.000552,0.000148,0.000049
3,"Large MLP $(128,128)$",6.148966e-07,0.004203,0.001194,0.000306
4,Medium MLP w/o input scaling,1.010814e-06,0.007988,0.000889,0.000578
5,Medium MLP w/o target scaling,1.621583e-06,0.005587,0.001976,0.000366


In [12]:
class ResidualOUSurrogate:
    """
    Baseline-aware residual surrogate:
        u_hat(t; u0, eps) = u_OU(t; u0) + eps * r_theta(u0, eps, t)

    The residual network learns:
        r_target = (u_true - u_OU) / eps
    """

    def __init__(self, hidden_layers=(64, 64), input_scaling=True, target_scaling=True, random_state=0):
        self.hidden_layers = hidden_layers
        self.input_scaling = input_scaling
        self.target_scaling = target_scaling
        self.random_state = random_state
        self.model = build_mlp(
            hidden_layers=hidden_layers,
            input_scaling=input_scaling,
            target_scaling=target_scaling,
            random_state=random_state,
        )

    @staticmethod
    def u_ou(X):
        """
        Unregularized Ornstein--Uhlenbeck variance solution:
            u_OU(t; u0) = 1 + (u0 - 1) exp(-2t)
        X columns: u0, eps, t
        """
        u0 = X[:, 0]
        t = X[:, 2]
        return 1.0 + (u0 - 1.0) * np.exp(-2.0 * t)

    def fit(self, X, y):
        eps = np.maximum(X[:, 1], 1e-12)
        y_residual = (y - self.u_ou(X)) / eps
        self.model.fit(X, y_residual)
        return self

    def predict(self, X):
        eps = X[:, 1]
        residual_pred = self.model.predict(X)
        return self.u_ou(X) + eps * residual_pred

In [14]:
# ------------------------------------------------------------
# Build generalization regimes for residual MLP evaluation
# ------------------------------------------------------------

# These ranges should match the regimes used for your existing Table 3.
# Adjust them if your previous notebook used different ranges.

interp_traj = generate_trajectory_dataset(
    u0_range=(0.2, 2.5),
    eps_range=(0.01, 0.5),
    n_param_pairs=200,
    t_grid=T_GRID,
    random_state=101,
)

low_u0_traj = generate_trajectory_dataset(
    u0_range=(0.05, 0.2),
    eps_range=(0.01, 0.5),
    n_param_pairs=100,
    t_grid=T_GRID,
    random_state=102,
)

high_u0_traj = generate_trajectory_dataset(
    u0_range=(2.5, 4.0),
    eps_range=(0.01, 0.5),
    n_param_pairs=100,
    t_grid=T_GRID,
    random_state=103,
)

high_eps_traj = generate_trajectory_dataset(
    u0_range=(0.2, 2.5),
    eps_range=(0.5, 0.8),
    n_param_pairs=100,
    t_grid=T_GRID,
    random_state=104,
)

NameError: name 'generate_trajectory_dataset' is not defined

In [13]:
# Train the mechanism-aware residual MLP
residual_model = ResidualOUSurrogate(
    hidden_layers=(64, 64),
    input_scaling=True,
    target_scaling=True,
    random_state=16,
)

residual_model.fit(X_train, y_train)

# Evaluate residual MLP across the same regimes used in Table 3
generalization_rows = []

regimes = [
    ("Interpolation", interp_traj),
    ("Low-$u_{0}$ extrapolation", low_u0_traj),
    ("High-$u_{0}$ extrapolation", high_u0_traj),
    ("High-$\\varepsilon$ extrapolation", high_eps_traj),
]

for regime_name, regime_data in regimes:
    scores = evaluate_model(residual_model, regime_data, T_GRID)

    generalization_rows.append([
        regime_name,
        scores["E_traj"],
        scores["E_cross"],
        scores["E_over"],
        scores["E_eq"],
    ])

residual_generalization_df = pd.DataFrame(
    generalization_rows,
    columns=["Test regime", "E_traj", "E_cross", "E_over", "E_eq"]
)

residual_generalization_df

NameError: name 'interp_traj' is not defined

## 6. Save CSV results

In [ ]:
csv_path = Path("mlp_surrogate_ablation_results.csv")
ablation_df.to_csv(csv_path, index=False)
print(f"Saved ablation results to: {csv_path.resolve()}")
ablation_df

## 7. Print LaTeX table for Overleaf

In [ ]:
def sci(x, digits=4):
    if pd.isna(x):
        return "N/A"
    s = f"{x:.{digits}e}"
    mantissa, exponent = s.split("e")
    exponent = int(exponent)
    return rf"${float(mantissa):.{digits}f}\times 10^{{{exponent}}}$"

print(r"""
\begin{table}[t]
\centering
\small
\caption{Ablation study of the MLP surrogate architecture and normalization strategy. Lower values indicate better performance.}
\label{tab:surrogate-ablation}
\begin{tabular}{lcccc}
\hline
Variant & $E_{\mathrm{traj}}$ & $E_{\mathrm{cross}}$ & $E_{\mathrm{over}}$ & $E_{\mathrm{eq}}$ \\
\hline
""")
for _, row in ablation_df.iterrows():
    print(
        f"{row['Variant']} & "
        f"{sci(row['E_traj'])} & "
        f"{sci(row['E_cross'])} & "
        f"{sci(row['E_over'])} & "
        f"{sci(row['E_eq'])} \\\\"
    )
print(r"""
\hline
\end{tabular}
\end{table}
""")

## 8. Suggested paper paragraph

Use this paragraph after inserting the table and adjust the interpretation based on the actual values:

Table~\ref{tab:surrogate-ablation} reports an ablation study of the MLP surrogate architecture and normalization strategy. The medium two-layer MLP provides the default configuration used in the main experiments. The ablation evaluates whether surrogate accuracy is driven primarily by model capacity or by stable input-output scaling. In general, the results show that both capacity and normalization affect mechanism-level prediction. Smaller networks may underfit the nonlinear transient dynamics, while larger networks provide limited additional improvement. Removing input or target normalization can degrade crossing-time and equilibrium prediction, indicating that stable scaling is important for learning mechanism-level quantities in the Fisher-regularized variance system.